In [2]:
import tensorflow as tf
import TensorSlider as ts
import keras


In [3]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
def createLabelsBatch(data, lookforward):
    """
    create input labels from the lookahead data
    """
    
    shape = data.shape
    
    # lookforward label making, format: batch, time (ascending from current), ohlcv
    divided = tf.divide(lookforward, tf.reshape(lookforward[:,0], (shape[0], 1, 5)))# divide lookforward by the base timeframe latest open
    # we expand dims because we are dividing each time the same
    divided -= 1 # zero out
    divided *= 10 # convert to 1/10th percentage, so 1 = 10 percent
    max = tf.reduce_max(divided[:,:,1], 1)
    min = tf.reduce_min(divided[:,:,2], 1)
    label = tf.stack([min, max], axis=1)
    
    # data processing, format: batch, timeframe, feature(ohlcv), window (ascending time)
    # prices
    dividepricesby = tf.reshape(data[:,0,3,-1], (shape[0], 1, 1, 1))
    prices = tf.divide(data[:,:,0:4], dividepricesby) # divide by latest base timeframe close
    prices -= 1 # zero out
    prices *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    volumes = tf.reshape(data[:,:,4], (shape[0], shape[1], 1, shape[3]))
    dividebyvolume = tf.reshape(data[:,:,4,-1], (shape[0], shape[1], 1, 1))
    volume = tf.divide(volumes, dividebyvolume) # divide by latest volume (of each timeframe)
    volume -= 1
    volume *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    data = tf.clip_by_value(tf.concat([prices, volume], axis=2), -2, 2)
    # remove nans
    data, label = tf.keras.ops.nan_to_num(data), tf.keras.ops.nan_to_num(label)

    return data, label

def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds

## Get Datasets

In [5]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 100
lookahead = 5
batch_size = 100

def getSlider(coin):

    path = tfrecordpath + coin+"/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return ts.WindowSlider(datasets, windowsize, lookahead, batch_size)

def getSliders(coins):
    datasets = []
    for coin in coins:
        datasets.append(tf.data.Dataset.from_generator(lambda: getSlider(coin),
            output_signature=(
                tf.TensorSpec((batch_size,5,5,windowsize), dtype=tf.float32),
                tf.TensorSpec((batch_size, lookahead+1,5), dtype=tf.float32))
            ))#.prefetch(tf.data.AUTOTUNE))
    return datasets

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP"]
datasets = getSliders(coins)

I0000 00:00:1738850968.148652   20220 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


## Combine Datasets

In [6]:
#ds = tf.data.Dataset.sample_from_datasets(datasets = datasets).prefetch(tf.data.AUTOTUNE)
ds = datasets[0]

ds = ds.map(createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [7]:
import keras
import os

def load_model(name):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    # Check if we have any checkpoints in the first place
    if not os.path.exists(folder + "checkpoints"):
        # No checkpoint folder, lets create one, and return the model, epoch 0
        os.mkdir(folder + "checkpoints")
        return model, 0

    # Check how many epoch checkpoints we have in the (existing!) checkpoint folder
    checkpoints = os.listdir("models/" + name + "/checkpoints/")

    # We do this by just counting how many files are in there, we assume there will be no vandalism
    # and all files inside the checkpoint folder are checkpoints
    lastEpoch = len(checkpoints)

    # load the latest checkpoint if we have more than 1 of them
    if lastEpoch >= 1:
        model.load_weights("models/" + name + "/checkpoints/" + str(lastEpoch) + ".weights.h5")

    return model, lastEpoch

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, name, lastEpoch=0):
        super().__init__()
        # no idea if we want to or need to super this
        self.lastEpoch = lastEpoch
        self.name = name
        print("Model was trained for " + str(lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.model.save_weights("models/" + str(self.name) + "/checkpoints/" + str(self.lastEpoch) + ".weights.h5")
        print("\nSaved checkpoint of epoch number " + str(self.lastEpoch))
        # Save the latest model
        self.model.save("models/" + str(self.name) + "/model.keras")


In [8]:
modelName = "modeltest"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=1,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )

model, epochs = load_model(modelName)

history = model.fit(ds, epochs=10, verbose=1, validation_data=None, callbacks=[saveEachEpoch(modelName, epochs), tensorboard])

2025-02-06 15:09:28.453137: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2025-02-06 15:09:28.453169: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.
2025-02-06 15:09:28.453272: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1006] Profiler found 1 GPUs
2025-02-06 15:09:28.497433: W external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1177] Fail to use per-thread activity buffer, cupti trace overhead may be big. CUPTI ERROR CODE:1
2025-02-06 15:09:28.497607: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:130] Profiler session tear down.
2025-02-06 15:09:28.497744: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1213] CUPTI activity buffer flushed


Model was trained for 0 epochs before.
Epoch 1/10


/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2025-02-06 15:09:30.826508: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
/mnt/c/Users/alexs/Desktop/levbot/Training/TensorSlider.py:112: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "init_from_zero" failed type inference due to: non-precise type pyobject
During: typing of argument at /mnt/c/Users/alexs/Desktop/levbot/Training/TensorSlider.py (120)

File "TensorSlider.py", line 120:
    def init_from_zero(self):
        <source elided>
        self.baseTensor = self.base.__iter__().__next__()
        for key, dataset in self.others.items():
        ^

  @jit(forceobj=

InvalidArgumentError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  ValueError: min() arg is an empty sequence
Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/data/ops/dataset_ops.py", line 865, in get_iterator
    return self._iterators[iterator_id]
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^

KeyError: 0


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot